In [1]:
import nltk
import pandas as pd
import ast

df = pd.read_csv('/Users/keremozemre/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/social compute/Social_computational_science_Project/politicians_full_text.csv')

In [2]:
import nltk
import re
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def tokenize(text):
    text = text.lower()
    
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    stems = [stemmer.stem(word) for word in filtered_words]
    
    
    return stems

df["tokens"] = df["body_text"].dropna().apply(tokenize)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/keremozemre/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [3]:
df["tokens"]

0       [stephen, n, miller, born, august, american, p...
1       [valeri, june, jarrett, ne, bowman, born, nove...
2       [cedric, levan, richmond, born, septemb, ameri...
3       [georg, herbert, walker, bush, june, novemb, s...
4       [david, axelrod, born, februari, american, pol...
                              ...                        
6133    [ann, wynia, ne, jobe, born, septemb, american...
6134    [bill, vasey, born, januari, american, politic...
6135    [catherin, lucil, hanaway, born, novemb, ameri...
6136    [dan, debicella, born, octob, former, state, s...
6137    [emili, e, gallagh, born, march, american, pol...
Name: tokens, Length: 6138, dtype: object

In [4]:
import re
import math
import numpy as np
from collections import Counter
from scipy.stats import chi2
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords, wordnet
import nltk

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def get_wordnet_pos(word):
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_map = {'J': wordnet.ADJ, 'V': wordnet.VERB, 'R': wordnet.ADV}
    return tag_map.get(tag, wordnet.NOUN)

def tokenize_lemmatized(abstract):
    """Clean → remove stopwords → lemmatize. Used for bigram discovery."""
    if not isinstance(abstract, str):
        return []
    text = abstract.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = [w for w in text.split() if w not in stop_words]
    return [lemmatizer.lemmatize(w, get_wordnet_pos(w)) for w in words]

df["tokens_lemmatized"] = df["body_text"].apply(tokenize_lemmatized)
df[["tokens", "tokens_lemmatized"]].head()

,tokens,tokens_lemmatized
0,"[stephen, n, miller, born, august, american, p...","[stephen, n, miller, born, august, american, p..."
1,"[valeri, june, jarrett, ne, bowman, born, nove...","[valerie, june, jarrett, ne, bowman, born, nov..."
2,"[cedric, levan, richmond, born, septemb, ameri...","[cedric, levan, richmond, born, september, ame..."
3,"[georg, herbert, walker, bush, june, novemb, s...","[george, herbert, walker, bush, june, november..."
4,"[david, axelrod, born, februari, american, pol...","[david, axelrod, born, february, american, pol..."


In [5]:
from nltk.probability import FreqDist

all_tokens = df["tokens_lemmatized"].explode().tolist()

fq = FreqDist(all_tokens)
fq.most_common(10)

[('state', 61110),
 ('republican', 32334),
 ('election', 32220),
 ('vote', 29227),
 ('house', 28566),
 ('serve', 27584),
 ('senate', 23686),
 ('u', 23575),
 ('district', 23052),
 ('would', 22606)]

In [6]:
clean_tokens = [t for t in all_tokens if t is not None and not (isinstance(t, float) and math.isnan(t))]

bigrams = list(nltk.bigrams(clean_tokens))

bigram_counts = Counter(bigrams)
w1_counts     = Counter([w1 for w1, _ in bigrams])
w2_counts     = Counter([w2 for _, w2 in bigrams])
N             = len(bigrams)

chi2_scores  = {}
p_values     = {}
co_occurences = {}

for (w1, w2), nii in bigram_counts.items():
    nio = w1_counts[w1] - nii
    noi = w2_counts[w2] - nii
    noo = N - (nii + nio + noi)

    observed = np.array([[nii, nio], [noi, noo]])
    row_sums  = observed.sum(axis=1)
    col_sums  = observed.sum(axis=0)
    expected  = np.outer(row_sums, col_sums) / N

    chi2_stat = ((observed - expected) ** 2 / expected).sum()
    p_val     = chi2.sf(chi2_stat, df=1)

    co_occurences[(w1, w2)] = nii
    chi2_scores[(w1, w2)]   = chi2_stat
    p_values[(w1, w2)]      = p_val

print(f"Total bigram types: {len(bigram_counts):,}")

Total bigram types: 2,093,929


In [7]:
filtered_bigrams = {
    bg:(co_occurences[bg]) for bg in bigram_counts
    if (p_values[bg] < 0.001) and (co_occurences[bg] > 50)
}

In [8]:
sorted_bigrams = sorted(
    filtered_bigrams.items(),
    key=lambda x: x[1],
    reverse=True
)
print(sorted_bigrams)

[(('united', 'state'), 15968), (('house', 'representative'), 9398), (('new', 'york'), 8682), (('general', 'election'), 6819), (('state', 'senate'), 5172), (('high', 'school'), 4596), (('congressional', 'district'), 4440), (('early', 'life'), 4025), (('attorney', 'general'), 3704), (('democratic', 'party'), 3699), (('supreme', 'court'), 3656), (('th', 'district'), 3579), (('republican', 'party'), 3390), (('new', 'jersey'), 3359), (('american', 'politician'), 3146), (('democratic', 'primary'), 3124), (('u', 'house'), 3102), (('u', 'senate'), 3059), (('personal', 'life'), 3040), (('lieutenant', 'governor'), 3003), (('republican', 'primary'), 2900), (('state', 'house'), 2877), (('state', 'senator'), 2789), (('donald', 'trump'), 2709), (('city', 'council'), 2697), (('life', 'education'), 2642), (('announce', 'would'), 2576), (('th', 'congressional'), 2461), (('white', 'house'), 2382), (('health', 'care'), 2343), (('presidential', 'election'), 2329), (('secretary', 'state'), 2212), (('politi

In [9]:
import nltk
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')

from nltk.tokenize import MWETokenizer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

lemmatizer = WordNetLemmatizer()


def get_wordnet_pos(word):
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_map = {'J': wordnet.ADJ, 'V': wordnet.VERB, 'R': wordnet.ADV}
    return tag_map.get(tag, wordnet.NOUN)  # default to NOUN

collocations  = list(filtered_bigrams.keys())
mwe_tokenizer = MWETokenizer(collocations, separator='_')

def tokenize(abstract):
    """
    Full pipeline:
      1. Lowercase & clean
      2. Remove stopwords
      3. Merge known collocations (MWE)
      4. Lemmatize single-word tokens (collocations left as-is)
    """
    if not isinstance(abstract, str):
        return []
    text = abstract.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    words = [w for w in text.split() if w not in stop_words]

    # Merge collocations first
    tokens = mwe_tokenizer.tokenize(words)

    # Lemmatize only single-word tokens; leave collocation tokens (contain '_') as-is
    lemmatized = [
        token if '_' in token else lemmatizer.lemmatize(token, get_wordnet_pos(token))
        for token in tokens
    ]
    return lemmatized

df["tokens"] = df["body_text"].apply(tokenize)

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/keremozemre/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/keremozemre/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [10]:
df["tokens"].head(5)

0    [stephen, n, miller, born_august, american_pol...
1    [valerie, june, jarrett, ne, bowman, born_nove...
2    [cedric, levan, richmond, born_september, amer...
3    [george, herbert, walker, bush, june, november...
4    [david, axelrod, born_february, american_polit...
Name: tokens, dtype: object

In [11]:
df.to_csv("tokenized_wikipage.csv",index = False)